In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PATH DEFINING

In [ ]:
#defining paths
import os

BASE_DIR = "/content/drive/MyDrive/multimodal_project"

VIDEO_DIR = os.path.join(
    BASE_DIR,
    "dataset",
    "subset_videos"
)

FRAME_DIR = os.path.join(
    BASE_DIR,
    "frames"
)

AUDIO_DIR = os.path.join(
    BASE_DIR,
    "audio"
)

IMAGE_EMB_DIR = os.path.join(
    BASE_DIR,
    "image_embeddings"
)

AUDIO_EMB_DIR = os.path.join(
    BASE_DIR,
    "audio_embeddings"
)

VIDEO_EMB_DIR = os.path.join(
    BASE_DIR,
    "video_embeddings"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "outputs"
)

print("Video Count:",
      len(os.listdir(VIDEO_DIR)))

LOADING VIDEOS WITH BOTH AUDIO AND FRAME

In [ ]:
#load common ids of 529 videos
import pandas as pd
import os

BASE_DIR = "/content/drive/MyDrive/multimodal_project"

OUTPUT_DIR = os.path.join(BASE_DIR,"outputs")

common_df = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "common_samples.csv"
    )
)

common_ids = common_df[
    "video_id"
].tolist()

print("Common IDs:", len(common_ids))

FULL VIDEO EMBEDDING CREATION

In [ ]:
#generate video embeddings using VideoMAE from hugging face
!pip install -q transformers

In [ ]:
##Load VideoMAE
from transformers import VideoMAEModel
from transformers import VideoMAEImageProcessor

processor_video = VideoMAEImageProcessor.from_pretrained(
    "MCG-NJU/videomae-base"
)

video_model = VideoMAEModel.from_pretrained(
    "MCG-NJU/videomae-base"
)

print("VideoMAE Loaded")

In [ ]:
#function for frame generator
import cv2
import numpy as np

def sample_frames(
    video_path,
    num_frames=16
):

    cap = cv2.VideoCapture(video_path)

    total_frames = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    indices = np.linspace(
        0,
        total_frames-1,
        num_frames,
        dtype=int
    )

    frames = []

    for idx in indices:

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            idx
        )

        success, frame = cap.read()

        if success:

            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            frames.append(frame)

    cap.release()

    return frames

In [ ]:
#testing on 5 frames from video
import torch
import numpy as np
import os

test_ids = common_ids[:5]

for vid in test_ids:

    try:

        video_path = os.path.join(
            VIDEO_DIR,
            vid + ".mp4"
        )

        frames = sample_frames(
            video_path,
            num_frames=16
        )

        inputs = processor_video(
            frames,
            return_tensors="pt"
        )

        with torch.no_grad():

            outputs = video_model(
                **inputs
            )

        # VideoMAE embedding
        embedding = outputs.last_hidden_state.mean(dim=1)

        save_path = os.path.join(
            VIDEO_EMB_DIR,
            vid + ".npy"
        )

        np.save(
            save_path,
            embedding.cpu().numpy()
        )

        print(
            f"Saved: {vid} -> {embedding.shape}"
        )

    except Exception as e:

        print(
            f"Failed: {vid}"
        )

        print(e)

print("Test Complete")

In [ ]:
#verify embeddings
import numpy as np
import os

sample = os.listdir(
    VIDEO_EMB_DIR
)[0]

emb = np.load(
    os.path.join(
        VIDEO_EMB_DIR,
        sample
    )
)

print(emb.shape)

FINAL ALL FRAMES

In [ ]:
#for all 529 videos
import torch
import numpy as np
import os

count = 0
failed = []

for vid in common_ids:

    try:

        save_path = os.path.join(
            VIDEO_EMB_DIR,
            vid + ".npy"
        )

        if os.path.exists(
            save_path
        ):
            continue

        video_path = os.path.join(
            VIDEO_DIR,
            vid + ".mp4"
        )

        frames = sample_frames(
            video_path,
            num_frames=16
        )

        inputs = processor_video(
            frames,
            return_tensors="pt"
        )

        with torch.no_grad():

            outputs = video_model(
                **inputs
            )

        embedding = outputs.last_hidden_state.mean(dim=1)

        np.save(
            save_path,
            embedding.cpu().numpy()
        )

        count += 1

        if count % 25 == 0:
            print(
                f"Processed {count}"
            )

    except Exception as e:

        failed.append(
            (vid, str(e))
        )

print("Saved:", count)
print("Failed:", len(failed))

In [ ]:
#verify final count
import os
print(
    "Video Embeddings:",
    len(os.listdir(VIDEO_EMB_DIR))
)

In [ ]:
#create traning dataset
import numpy as np

IMAGE_EMB_DIR = os.path.join(
    BASE_DIR,
    "image_embeddings"
)

AUDIO_EMB_DIR = os.path.join(
    BASE_DIR,
    "audio_embeddings"
)

X = []
Y = []

for vid in common_ids:

    video_path = os.path.join(
        VIDEO_EMB_DIR,
        vid + ".npy"
    )

    if not os.path.exists(
        video_path
    ):
        continue

    img_emb = np.load(
        os.path.join(
            IMAGE_EMB_DIR,
            vid + ".npy"
        )
    ).squeeze()

    aud_emb = np.load(
        os.path.join(
            AUDIO_EMB_DIR,
            vid + ".npy"
        )
    )

    vid_emb = np.load(
        video_path
    ).squeeze()

    fusion_input = np.concatenate(
        [
            img_emb,
            aud_emb
        ]
    )

    X.append(
        fusion_input
    )

    Y.append(
        vid_emb
    )

X = np.array(X)
Y = np.array(Y)

print(X.shape)
print(Y.shape)

In [ ]:
#save dataset
np.save(
    os.path.join(
        OUTPUT_DIR,
        "X.npy"
    ),
    X
)

np.save(
    os.path.join(
        OUTPUT_DIR,
        "Y.npy"
    ),
    Y
)

print("Dataset Saved")

CREATING TRAIN,TEST,VAL DATASET

In [ ]:
#train/test/val split
from sklearn.model_selection import train_test_split

X_train, X_temp, Y_train, Y_temp = train_test_split(
    X,
    Y,
    test_size=0.30,
    random_state=42
)

X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp,
    Y_temp,
    test_size=0.50,
    random_state=42
)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

In [ ]:
#save splits
np.save(f"{OUTPUT_DIR}/X_train.npy", X_train)
np.save(f"{OUTPUT_DIR}/X_val.npy", X_val)
np.save(f"{OUTPUT_DIR}/X_test.npy", X_test)

np.save(f"{OUTPUT_DIR}/Y_train.npy", Y_train)
np.save(f"{OUTPUT_DIR}/Y_val.npy", Y_val)
np.save(f"{OUTPUT_DIR}/Y_test.npy", Y_test)

In [ ]:
#create dataset class
import torch
from torch.utils.data import Dataset

class EmbeddingDataset(Dataset):

    def __init__(self, X, Y):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.Y = torch.tensor(
            Y,
            dtype=torch.float32
        )

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return (
            self.X[idx],
            self.Y[idx]
        )

In [ ]:
#Create DataLoaders
from torch.utils.data import DataLoader

train_dataset = EmbeddingDataset(
    X_train,
    Y_train
)

val_dataset = EmbeddingDataset(
    X_val,
    Y_val
)

test_dataset = EmbeddingDataset(
    X_test,
    Y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32
)